[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/ch08_03_attention_from_scratch.ipynb)

# Chapter 8-3: Attention 메커니즘 밑바닥부터 구현하기

> **목표**: Attention(어텐션)의 수학적 원리를 완전히 이해하고, NumPy로 직접 구현한 뒤, PyTorch Seq2Seq + Attention까지 만들어봅니다.
>
> **대상**: 딥러닝을 처음 공부하는 분 (RNN/LSTM 기초를 알면 좋습니다)
>
> **선수 지식**: ch08_01 (RNN), ch08_02 (LSTM) 노트북을 먼저 보시면 좋습니다.
>
> **소요 시간**: 약 3~4시간

---

### 한 줄 비유로 이해하는 Attention

> Attention은 **시험 공부할 때 중요한 부분에 형광펜을 치는 것**과 같습니다.
> 모든 내용을 똑같이 외우는 대신, "지금 풀어야 할 문제"와 관련된 부분에 집중하는 것이죠.

---

## 목차

| 파트 | 내용 | 핵심 |
|------|------|------|
| **Part 1** | Attention 이론 | RNN 병목(bottleneck), Bahdanau/Luong Attention 수식 완전 분해, Self-Attention 소개 |
| **Part 2** | NumPy 밑바닥 구현 | Additive/Dot-Product Attention 클래스, 히트맵(heatmap) 시각화 |
| **Part 3** | PyTorch Seq2Seq + Attention | Bidirectional LSTM Encoder, Bahdanau Decoder, 감정 분류 |
| **Part 4** | 정리 | Attention 한계, Transformer 동기, 핵심 수식 요약 |

In [ ]:
# ============================================================
# Colab 환경 설정 셀
# Google Colab에서 실행할 때 필요한 패키지를 설치합니다.
# 로컬 환경에서는 이미 설치되어 있다면 건너뛰어도 됩니다.
# ============================================================

# numpy: 행렬 연산을 위한 핵심 라이브러리 (Attention 밑바닥 구현에 사용)
# matplotlib: Attention 히트맵 등을 시각화하기 위한 그래프 라이브러리
# seaborn: matplotlib 위에 얹는 고급 시각화 라이브러리 (히트맵에 사용)
# torch: PyTorch - Part 3에서 Seq2Seq + Attention 구현에 사용
!pip install -q numpy matplotlib seaborn torch

# Colab에서 한글 폰트를 사용하기 위한 설정
import sys
if 'google.colab' in sys.modules:
    !apt-get -qq install -y fonts-nanum > /dev/null 2>&1
    import matplotlib.font_manager as fm
    fm._rebuild()
    print("Colab 한글 폰트(나눔고딕) 설치 완료!")

In [ ]:
# ============================================================
# 공통 임포트
# 이 노트북 전체에서 사용할 라이브러리를 한 번에 불러옵니다.
# ============================================================

import numpy as np                # 행렬/벡터 연산의 기본 라이브러리
import matplotlib.pyplot as plt   # 그래프/시각화 라이브러리
import seaborn as sns             # 히트맵 시각화를 위한 라이브러리
import torch                      # PyTorch 딥러닝 프레임워크
import torch.nn as nn             # PyTorch 신경망 모듈
import torch.nn.functional as F   # 활성화 함수 등 유틸리티
import warnings                   # 불필요한 경고 메시지 숨기기
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Colab과 로컬 환경 모두 지원)
import sys
if 'google.colab' in sys.modules:
    # Colab: 나눔고딕 폰트 사용
    plt.rcParams['font.family'] = 'NanumGothic'
else:
    # macOS: AppleGothic 폰트 사용
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False      # 마이너스 기호 깨짐 방지

# 재현성을 위한 시드(seed) 고정
# 시드를 고정하면 난수 생성이 매번 동일한 결과를 내므로, 실험 결과를 재현할 수 있습니다.
np.random.seed(42)         # NumPy 난수 시드 고정
torch.manual_seed(42)      # PyTorch 난수 시드 고정

print("모든 라이브러리가 정상적으로 로드되었습니다!")  # 확인 메시지 출력

---

# Part 1: Attention 이론

> **이 파트에서 배울 것**: Attention이 왜 필요한지, 어떤 수식으로 동작하는지를 완전히 이해합니다.

---

## 1.1 RNN/LSTM의 정보 병목(Information Bottleneck) 문제

### 기존 Seq2Seq(시퀀스-투-시퀀스) 모델의 구조

기계 번역을 예로 들어봅시다. "나는 학생입니다"를 "I am a student"로 번역하려면:

```
[Encoder(인코더)]                         [Decoder(디코더)]
나는 → h₁ → 학생 → h₂ → 입니다 → h₃ ──→ I → am → a → student
                                  ↑
                          컨텍스트 벡터 c = h₃
```

**문제점: 정보 병목 (Information Bottleneck)**

쉽게 말하면, **긴 책의 내용을 메모지 한 장에 요약하라**는 것과 같습니다.
Encoder(인코더)는 입력 문장 전체를 **마지막 은닉 상태(hidden state) $h_T$ 하나**에 압축해야 합니다.

| 문제 | 설명 |
|------|------|
| **긴 문장** | "나는 어제 친구와 함께 서울역 근처 카페에서 커피를 마시며 오랜만에 이야기를 나누었다" → 이 긴 문장의 모든 정보를 벡터 하나에? |
| **정보 손실** | 앞쪽 단어 "나는"의 정보는 $h_1$에 담겼지만, $h_T$까지 전달되면서 점점 희석(diluted)됨 |
| **기울기 소실(Vanishing Gradient)** | BPTT(시간에 따른 역전파)에서 긴 시퀀스의 앞쪽 단어까지 기울기가 잘 전파되지 않음 |

### 실험적 증거

Cho et al. (2014)의 실험에서, 기존 Seq2Seq 모델은 **문장 길이가 20을 넘어가면** BLEU(번역 품질 점수) 점수가 급격히 떨어졌습니다.

> **핵심 질문**: Decoder가 출력 단어를 생성할 때, Encoder의 **모든 시점**의 정보를 직접 참조할 수 있으면 어떨까?
>
> 이것이 바로 **Attention**의 핵심 아이디어입니다!

## 1.2 Seq2Seq 모델 복습 (Encoder-Decoder 구조)

Attention을 이해하려면, 먼저 기본 Seq2Seq 구조를 정확히 알아야 합니다.

### Encoder (인코더) - 입력 문장을 읽는 역할

입력 시퀀스 $x = (x_1, x_2, \ldots, x_{T_x})$를 받아 각 시점의 은닉 상태를 생성합니다.

$$h_i = \text{RNN}_{\text{enc}}(x_i, h_{i-1}), \quad i = 1, 2, \ldots, T_x$$

쉽게 말하면, **책을 한 단어씩 읽으면서 메모($h_i$)를 업데이트**하는 과정입니다.

- $x_i \in \mathbb{R}^{d_{\text{emb}}}$: $i$번째 입력 단어의 임베딩(embedding) 벡터 — 단어를 숫자 벡터로 변환한 것
- $h_i \in \mathbb{R}^{d_h}$: $i$번째 시점의 은닉 상태(hidden state) 벡터 — 지금까지 읽은 내용의 "요약 메모"
- $T_x$: 입력 시퀀스의 길이
- $d_{\text{emb}}$: 임베딩 차원, $d_h$: 은닉 상태 차원

### Decoder (디코더) - 출력 문장을 생성하는 역할

기존 Seq2Seq에서는 **컨텍스트 벡터(context vector)** $c = h_{T_x}$ (Encoder의 마지막 은닉 상태)만 사용합니다.

$$s_t = \text{RNN}_{\text{dec}}(y_{t-1}, s_{t-1}), \quad s_0 = c$$

$$P(y_t | y_{<t}, x) = \text{softmax}(W_o s_t + b_o)$$

- $s_t \in \mathbb{R}^{d_s}$: Decoder의 $t$번째 시점 은닉 상태
- $y_{t-1}$: 이전 시점에서 생성한 출력 단어
- $W_o \in \mathbb{R}^{|V| \times d_s}$: 출력 변환 행렬 ($|V|$는 어휘 크기)

### 문제의 핵심

$$c = h_{T_x}$$

이 한 줄이 모든 문제의 원인입니다. 아무리 긴 문장이라도 **고정 크기 벡터 하나**에 담아야 하니까요.

쉽게 말하면, 시험 공부한 모든 내용을 **포스트잇 한 장에 적어서** 시험장에 가져가는 것과 같습니다. 당연히 정보가 빠지겠죠!

## 1.3 Bahdanau Attention (Additive Attention, 가산 어텐션)

> **논문**: Bahdanau, Cho & Bengio (2015), "Neural Machine Translation by Jointly Learning to Align and Translate"

Bahdanau의 핵심 아이디어: Decoder가 각 출력 단어를 생성할 때, **Encoder의 모든 은닉 상태를 가중합(weighted sum)**하여 컨텍스트 벡터를 만들자!

쉽게 말하면, **포스트잇 한 장 대신 노트 전체를 펼쳐놓고**, 지금 필요한 부분을 골라 읽는 것입니다.

### Attention 메커니즘 전체 흐름 다이어그램

```
┌─────────────────────────────────────────────────────────────────┐
│                    Bahdanau Attention 흐름도                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Encoder 은닉 상태:  h₁    h₂    h₃    (입력: "나는 학생 입니다")  │
│                      │     │     │                              │
│                      ▼     ▼     ▼                              │
│  Step 1 (Score):   e₁    e₂    e₃   ← score(s_{t-1}, h_i)     │
│                      │     │     │                              │
│                      ▼     ▼     ▼                              │
│  Step 2 (Align):   α₁    α₂    α₃   ← softmax(e₁, e₂, e₃)   │
│                      │     │     │       합 = 1.0               │
│                      ▼     ▼     ▼                              │
│  Step 3 (Context): c_t = α₁·h₁ + α₂·h₂ + α₃·h₃  (가중합)     │
│                      │                                          │
│                      ▼                                          │
│  Decoder:  s_t = RNN(y_{t-1}, s_{t-1}, c_t) → 출력 단어 예측    │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Step 1: Score (에너지, energy) 계산

$$e_{t,i} = v^T \tanh(W_s s_{t-1} + W_h h_i)$$

이 수식을 **완전 분해**해봅시다:

| 기호 | 차원 | 의미 |
|------|------|------|
| $s_{t-1}$ | $\mathbb{R}^{d_s}$ | Decoder의 **이전 시점** 은닉 상태 (현재 무엇을 생성하려는지 담고 있음) |
| $h_i$ | $\mathbb{R}^{d_h}$ | Encoder의 $i$번째 시점 은닉 상태 (입력 문장의 $i$번째 단어 주변 정보) |
| $W_s$ | $\mathbb{R}^{d_a \times d_s}$ | Decoder 은닉 상태를 attention 공간으로 변환하는 가중치 행렬 |
| $W_h$ | $\mathbb{R}^{d_a \times d_h}$ | Encoder 은닉 상태를 attention 공간으로 변환하는 가중치 행렬 |
| $v$ | $\mathbb{R}^{d_a}$ | attention 공간의 벡터를 스칼라(scalar, 하나의 숫자) 점수로 변환하는 가중치 벡터 |
| $d_a$ | 스칼라 | attention 은닉 차원 (하이퍼파라미터) |
| $e_{t,i}$ | 스칼라 | Decoder 시점 $t$에서 Encoder 시점 $i$에 대한 "관련도 점수" |

**계산 과정을 단계별로 풀어보면:**

1. $W_s s_{t-1}$: $(d_a \times d_s) \cdot (d_s) = (d_a)$ → Decoder 상태를 $d_a$ 차원으로 변환
2. $W_h h_i$: $(d_a \times d_h) \cdot (d_h) = (d_a)$ → Encoder 상태를 $d_a$ 차원으로 변환
3. $W_s s_{t-1} + W_h h_i$: $(d_a) + (d_a) = (d_a)$ → 두 벡터를 **같은 공간에서 더함** (Additive!)
4. $\tanh(\cdot)$: $(d_a) \to (d_a)$ → 비선형 변환 (값을 $[-1, 1]$로 제한)
5. $v^T \cdot$: $(d_a)^T \cdot (d_a) = 1$ → 내적으로 스칼라 점수 하나를 얻음

> **왜 "Additive(가산)"인가?**: Step 3에서 두 벡터를 **더하기(+)** 때문입니다. 이것이 Luong의 "Multiplicative(곱셈)" Attention과의 핵심 차이입니다.

### Step 2: Alignment (정렬 가중치)

$$\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_{k=1}^{T_x} \exp(e_{t,k})}$$

이것은 **softmax(소프트맥스)** 함수입니다! 점수를 확률로 바꿔주는 역할을 합니다.

| 성질 | 설명 |
|------|------|
| $0 \leq \alpha_{t,i} \leq 1$ | 각 가중치는 0과 1 사이 |
| $\sum_{i=1}^{T_x} \alpha_{t,i} = 1$ | 모든 가중치의 합은 1 (확률 분포) |
| 해석 | $\alpha_{t,i}$는 "Decoder 시점 $t$에서 Encoder 시점 $i$에 **얼마나 주의(attention)를 기울일지**"의 확률 |

쉽게 말하면, **관심의 비율을 정하는 것**입니다. "나는"에 85%, "학생"에 10%, "입니다"에 5%처럼요.

**직관적 예시**: "나는 학생입니다" → "I"를 생성할 때
- $\alpha_{1, \text{나는}}$ = 0.85 (높음! "나는" → "I"이니까)
- $\alpha_{1, \text{학생}}$ = 0.10
- $\alpha_{1, \text{입니다}}$ = 0.05

### Step 3: Context Vector (컨텍스트 벡터)

$$c_t = \sum_{i=1}^{T_x} \alpha_{t,i} \, h_i$$

| 기호 | 차원 | 의미 |
|------|------|------|
| $c_t$ | $\mathbb{R}^{d_h}$ | Decoder 시점 $t$에서 사용할 **동적 컨텍스트 벡터** |
| $\alpha_{t,i}$ | 스칼라 | Encoder 시점 $i$에 대한 가중치 |
| $h_i$ | $\mathbb{R}^{d_h}$ | Encoder 시점 $i$의 은닉 상태 |

**핵심**: 기존 Seq2Seq는 $c$가 고정이었지만, Attention에서는 **$c_t$가 매 시점 달라집니다!**

쉽게 말하면, 기존 방식은 항상 같은 메모를 보지만, Attention은 **질문에 따라 다른 페이지를 펼쳐 보는 것**입니다.

### 최종 출력

컨텍스트 벡터 $c_t$와 Decoder 은닉 상태 $s_t$를 결합하여 출력을 생성합니다:

$$\tilde{s}_t = \tanh(W_c [c_t ; s_t])$$

$$P(y_t | y_{<t}, x) = \text{softmax}(W_o \tilde{s}_t)$$

- $[c_t ; s_t]$: $c_t$와 $s_t$의 **연결(concatenation)**, 차원은 $(d_h + d_s)$
- $W_c \in \mathbb{R}^{d_s \times (d_h + d_s)}$: 결합 벡터를 변환하는 행렬
- $\tilde{s}_t \in \mathbb{R}^{d_s}$: attention이 적용된 최종 은닉 상태

## 1.4 Luong Attention (Multiplicative Attention, 곱셈 어텐션)

> **논문**: Luong, Pham & Manning (2015), "Effective Approaches to Attention-based Neural Machine Translation"

Luong은 Bahdanau Attention을 **더 간단하고 다양한 방식**으로 일반화했습니다.

### 핵심 차이: Score 함수의 종류

Luong은 3가지 score 함수를 제안했습니다:

#### (1) Dot-Product Attention (내적 어텐션)

$$e_{t,i} = s_t^T h_i$$

| 기호 | 차원 | 설명 |
|------|------|------|
| $s_t$ | $\mathbb{R}^{d}$ | Decoder의 **현재** 시점 은닉 상태 (Bahdanau는 $s_{t-1}$을 썼다는 점에 주의!) |
| $h_i$ | $\mathbb{R}^{d}$ | Encoder의 $i$번째 은닉 상태 |
| $e_{t,i}$ | 스칼라 | 두 벡터의 내적(dot product) = 유사도 |

쉽게 말하면, 두 벡터가 **같은 방향을 가리킬수록 내적 값이 크다** → "관련이 높다"는 뜻입니다.

- **장점**: 학습할 파라미터가 없음! 가장 빠르고 단순
- **전제 조건**: $d_s = d_h$여야 함 (Decoder와 Encoder 은닉 차원이 같아야 내적이 가능)

#### (2) General Attention (일반 어텐션)

$$e_{t,i} = s_t^T W_a h_i$$

| 기호 | 차원 | 설명 |
|------|------|------|
| $W_a$ | $\mathbb{R}^{d_s \times d_h}$ | Encoder 은닉 상태를 Decoder 공간으로 변환하는 가중치 행렬 |

- **장점**: $d_s \neq d_h$여도 사용 가능 ($W_a$가 차원을 맞춰줌)
- **학습 파라미터**: $d_s \times d_h$개
- **직관**: $W_a h_i$는 "Encoder 상태를 Decoder 관점에서 재해석"한 것

#### (3) Concat Attention (연결 어텐션)

$$e_{t,i} = v^T \tanh(W_a [s_t ; h_i])$$

| 기호 | 차원 | 설명 |
|------|------|------|
| $[s_t ; h_i]$ | $\mathbb{R}^{d_s + d_h}$ | 두 벡터를 연결(concatenation) |
| $W_a$ | $\mathbb{R}^{d_a \times (d_s + d_h)}$ | 연결 벡터를 attention 공간으로 변환 |
| $v$ | $\mathbb{R}^{d_a}$ | 스칼라 점수로 변환하는 벡터 |

- 이것은 사실 Bahdanau Attention과 매우 유사합니다!
- 차이점: Bahdanau는 $W_s s_{t-1} + W_h h_i$ (별도 행렬로 변환 후 더하기), Luong Concat은 $W_a [s_t; h_i]$ (먼저 연결 후 하나의 행렬로 변환)

### Bahdanau vs Luong 비교 요약

| 항목 | Bahdanau (Additive, 가산) | Luong (Multiplicative, 곱셈) |
|------|---------------------|----------------------|
| Score 입력 | $s_{t-1}$ (이전 시점) | $s_t$ (현재 시점) |
| Score 함수 | $v^T \tanh(W_s s_{t-1} + W_h h_i)$ | dot / general / concat 중 택 1 |
| 연산 비용 | 상대적으로 높음 (행렬 2개 + tanh) | dot이 가장 빠름 |
| Encoder | 양방향(bidirectional) RNN 사용 | 단방향도 가능 |
| 컨텍스트 활용 | $c_t$를 다음 RNN 입력에 사용 | $c_t$와 $s_t$를 concat하여 출력에 사용 |

## 1.5 Self-Attention(자기 어텐션) 소개 — Q, K, V 개념 도입

지금까지 본 Attention은 **Cross-Attention(교차 어텐션)**입니다:
- Query(질의) = Decoder 상태
- Key(키)/Value(값) = Encoder 상태

**Self-Attention(자기 어텐션)**은 **같은 시퀀스 안에서** 각 위치가 다른 위치를 참조합니다.

쉽게 말하면, Cross-Attention은 "다른 사람의 노트를 참고하는 것"이고, Self-Attention은 "내 노트 안에서 관련 내용을 찾는 것"입니다.

### Q, K, V (Query, Key, Value)

Attention을 일반화하면 3가지 역할이 있습니다:

| 역할 | 의미 | 비유: 도서관 검색 |
|------|------|------|
| **Query (Q, 질의)** | "나는 무엇을 찾고 있는가?" | 도서관에서 검색하는 질문 |
| **Key (K, 키)** | "나는 어떤 정보를 가지고 있는가?" | 책의 제목/키워드 |
| **Value (V, 값)** | "실제로 제공할 정보" | 책의 실제 내용 |

### Self-Attention 수식

입력 시퀀스 $X = (x_1, x_2, \ldots, x_T)$에 대해:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

| 기호 | 차원 | 의미 |
|------|------|------|
| $X$ | $\mathbb{R}^{T \times d_{\text{model}}}$ | 입력 시퀀스 (T개 토큰, 각 $d_{\text{model}}$ 차원) |
| $W_Q, W_K$ | $\mathbb{R}^{d_{\text{model}} \times d_k}$ | Query, Key 변환 행렬 |
| $W_V$ | $\mathbb{R}^{d_{\text{model}} \times d_v}$ | Value 변환 행렬 |
| $QK^T$ | $\mathbb{R}^{T \times T}$ | 모든 위치 쌍의 유사도 행렬 |
| $\sqrt{d_k}$ | 스칼라 | 스케일링 팩터(scaling factor) — 내적 값이 너무 커지는 것을 방지 |

> **$\sqrt{d_k}$로 나누는 이유**: $d_k$가 크면 내적 값의 분산이 $d_k$에 비례하여 커집니다.
> softmax에 큰 값이 들어가면 기울기가 거의 0이 되어 학습이 안 됩니다.
> $\sqrt{d_k}$로 나누면 분산이 1로 정규화됩니다.
>
> 쉽게 말하면, 시험 점수를 100점 만점으로 **스케일을 맞추는 것**과 같습니다.

### Cross-Attention vs Self-Attention 비교

```
Cross-Attention (Seq2Seq):
  Q = Decoder 상태  →  "번역 중인 단어가 원문의 어디를 볼까?"
  K, V = Encoder 상태

Self-Attention (Transformer):
  Q, K, V = 같은 시퀀스  →  "이 문장 안에서 각 단어가 다른 단어와 어떤 관계?"
```

> Self-Attention은 다음 챕터(Transformer)에서 본격적으로 다룹니다. 여기서는 개념만 잡아두세요!

## 1.6 Attention의 직관적 의미: "어디를 볼 것인가?"

Attention의 핵심을 한 문장으로 요약하면:

> **"출력을 생성할 때, 입력의 어느 부분에 주의를 집중할 것인가?"**

### 일상 비유

| 상황 | Query (질의) | Key (키) | Value (값) | Attention (주의 집중) |
|------|-------|-----|-------|-----------|
| 시험 공부 | "이 문제의 답은?" | 교과서 각 챕터 제목 | 교과서 각 챕터 내용 | 관련 챕터를 집중해서 읽기 |
| 회의록 작성 | "결론은?" | 회의 발언 각각의 주제 | 발언 내용 전체 | 결론 관련 발언에 집중 |
| 번역 | "이 영어 단어에 대응하는 한국어는?" | 원문의 각 단어 | 원문의 각 단어 표현 | 대응하는 원문 단어에 집중 |

### 시각적 이해

"나는 학생입니다" → "I am a student" 번역에서의 Attention 가중치:

```
              나는    학생    입니다
    I         [0.85]  [0.10]  [0.05]    ← "I"를 생성할 때 "나는"에 집중
    am        [0.15]  [0.10]  [0.75]    ← "am"을 생성할 때 "입니다"에 집중  
    a         [0.05]  [0.15]  [0.80]    ← "a"를 생성할 때 "입니다"에 집중
    student   [0.05]  [0.90]  [0.05]    ← "student"를 생성할 때 "학생"에 집중
```

각 행의 합은 1이 됩니다 (softmax). 이 행렬을 **히트맵(heatmap)**으로 시각화하면 Attention이 어디를 보는지 직관적으로 알 수 있습니다.

---

> **Part 1 정리**: 이제 Attention의 이론을 완전히 이해했습니다. Part 2에서 이것을 NumPy로 직접 구현해봅시다!

---

# Part 2: Attention 밑바닥 구현 (NumPy only)

> **이 파트에서 배울 것**: Additive(가산) Attention과 Dot-Product(내적) Attention을 NumPy만으로 구현합니다.
> 모든 코드 라인에 한글 주석이 있습니다.

---

## 2.1 Softmax(소프트맥스) 함수 구현

Attention에서 가장 기본이 되는 softmax 함수부터 만들어봅시다.

softmax는 **임의의 점수 벡터를 확률 분포로 변환**하는 함수입니다.
쉽게 말하면, [2.0, 1.0, 0.1] 같은 점수를 [0.66, 0.24, 0.10] 같은 "합이 1인 비율"로 바꿔줍니다.

In [ ]:
def softmax(x):
    """안정적인 softmax 함수를 구현합니다.
    
    큰 값이 들어오면 exp()가 무한대(overflow)가 될 수 있으므로,
    최대값을 빼서 수치적 안정성을 확보합니다.
    
    왜 최대값을 빼도 되는가?
    수학적으로 softmax(x) = softmax(x - c) 가 성립합니다. (c는 상수)
    증명: exp(x_i - c) / sum(exp(x_j - c)) = exp(x_i) * exp(-c) / (sum(exp(x_j)) * exp(-c))
    
    Args:
        x: 입력 배열 (1D 또는 2D numpy 배열)
    
    Returns:
        softmax가 적용된 배열 (각 행의 합이 1)
    """
    # 수치 안정성을 위해 최대값을 뺍니다 (overflow 방지)
    # axis=-1: 마지막 축(행 방향)을 기준으로 최대값 계산
    # keepdims=True: 브로드캐스팅을 위해 차원 유지
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    
    # 각 행의 합으로 나누어 확률 분포로 변환합니다
    return e_x / np.sum(e_x, axis=-1, keepdims=True)


# === softmax 함수 동작 테스트 ===
# 테스트용 점수 배열 생성
test_scores = np.array([2.0, 1.0, 0.1])  # 3개의 점수

# softmax 적용
test_probs = softmax(test_scores)  # 점수를 확률 분포로 변환

# 결과 출력
print("입력 점수:", test_scores)        # 원래 점수값
print("softmax 결과:", test_probs)       # 변환된 확률값
print("합계:", np.sum(test_probs))       # 합이 1인지 확인
print()  # 빈 줄 출력

# 2D 배열에서도 동작하는지 테스트 (Attention에서는 배치 처리 시 2D를 사용)
test_2d = np.array([[2.0, 1.0, 0.1],    # 첫 번째 행
                    [1.0, 2.0, 3.0]])     # 두 번째 행
print("2D softmax 결과:")                 # 2D 배열 테스트
print(softmax(test_2d))                   # 각 행마다 독립적으로 softmax 적용
print("각 행의 합:", np.sum(softmax(test_2d), axis=1))  # 각 행의 합이 1인지 확인

## 2.2 Additive Attention (Bahdanau) 클래스 구현

Bahdanau Attention의 수식을 그대로 NumPy 코드로 옮깁니다:

$$e_{t,i} = v^T \tanh(W_s s_{t-1} + W_h h_i)$$
$$\alpha_{t,i} = \text{softmax}(e_{t,i})$$
$$c_t = \sum_i \alpha_{t,i} h_i$$

> 아래 코드에서 수식의 각 부분이 어떻게 코드로 변환되는지 **주석을 따라 읽어보세요**.
> 특히 행렬 곱의 차원 변화를 단계별로 추적하는 것이 중요합니다.

In [ ]:
class AdditiveAttention:
    """Bahdanau(Additive) Attention을 NumPy로 구현한 클래스입니다.
    
    수식: e_{t,i} = v^T * tanh(W_s * s_{t-1} + W_h * h_i)
    
    Args:
        d_s: Decoder 은닉 상태의 차원 (정수)
        d_h: Encoder 은닉 상태의 차원 (정수)
        d_a: Attention 은닉 차원 (정수, 하이퍼파라미터)
    """
    
    def __init__(self, d_s, d_h, d_a):
        """Attention에 필요한 가중치 행렬들을 초기화합니다."""
        # W_s: Decoder 상태를 attention 공간으로 변환하는 행렬
        # 크기: (d_a, d_s) - d_s 차원을 d_a 차원으로 변환
        # Xavier 초기화: 표준편차를 sqrt(2 / (입력+출력))으로 설정
        self.W_s = np.random.randn(d_a, d_s) * np.sqrt(2.0 / (d_a + d_s))
        
        # W_h: Encoder 상태를 attention 공간으로 변환하는 행렬
        # 크기: (d_a, d_h) - d_h 차원을 d_a 차원으로 변환
        self.W_h = np.random.randn(d_a, d_h) * np.sqrt(2.0 / (d_a + d_h))
        
        # v: attention 공간의 벡터를 스칼라 점수로 변환하는 가중치 벡터
        # 크기: (d_a,) - d_a 차원 벡터를 스칼라 1개로 변환
        self.v = np.random.randn(d_a) * np.sqrt(2.0 / d_a)
        
        # 차원 정보를 저장합니다 (디버깅용)
        self.d_s = d_s  # Decoder 은닉 차원
        self.d_h = d_h  # Encoder 은닉 차원
        self.d_a = d_a  # Attention 은닉 차원
    
    def score(self, s_prev, h_i):
        """하나의 Encoder 은닉 상태에 대한 attention score를 계산합니다.
        
        수식: e = v^T * tanh(W_s * s_prev + W_h * h_i)
        
        Args:
            s_prev: Decoder의 이전 시점 은닉 상태 (d_s,)
            h_i: Encoder의 i번째 은닉 상태 (d_h,)
        
        Returns:
            e: 스칼라 attention score
        """
        # Step 1: W_s @ s_prev → Decoder 상태를 attention 공간으로 변환
        # (d_a, d_s) @ (d_s,) = (d_a,)
        proj_s = self.W_s @ s_prev
        
        # Step 2: W_h @ h_i → Encoder 상태를 attention 공간으로 변환
        # (d_a, d_h) @ (d_h,) = (d_a,)
        proj_h = self.W_h @ h_i
        
        # Step 3: 두 변환 결과를 더합니다 (Additive!)
        # (d_a,) + (d_a,) = (d_a,)
        added = proj_s + proj_h
        
        # Step 4: tanh 활성화 함수 적용 (값을 [-1, 1]로 제한)
        # (d_a,) → (d_a,)
        activated = np.tanh(added)
        
        # Step 5: v와 내적하여 스칼라 점수를 얻습니다
        # (d_a,) . (d_a,) = 스칼라
        e = self.v @ activated
        
        return e  # 스칼라 attention score 반환
    
    def forward(self, s_prev, encoder_outputs):
        """Additive Attention의 전체 순전파를 수행합니다.
        
        Args:
            s_prev: Decoder의 이전 시점 은닉 상태 (d_s,)
            encoder_outputs: Encoder의 모든 은닉 상태 (T_x, d_h)
                T_x = 입력 시퀀스 길이
        
        Returns:
            context: 컨텍스트 벡터 (d_h,)
            attention_weights: attention 가중치 (T_x,) - 합이 1인 확률 분포
        """
        # 입력 시퀀스 길이를 가져옵니다
        T_x = encoder_outputs.shape[0]  # Encoder 시점 개수
        
        # 각 Encoder 시점에 대한 score를 저장할 배열
        scores = np.zeros(T_x)  # (T_x,) 크기의 빈 배열
        
        # === Step 1: 모든 Encoder 시점에 대한 score 계산 ===
        for i in range(T_x):  # 각 Encoder 시점에 대해 반복
            # i번째 Encoder 은닉 상태와의 score 계산
            scores[i] = self.score(s_prev, encoder_outputs[i])
        
        # === Step 2: softmax로 정렬 가중치(alignment weights) 계산 ===
        # scores를 확률 분포로 변환 (합이 1)
        attention_weights = softmax(scores)  # (T_x,)
        
        # === Step 3: 가중합으로 컨텍스트 벡터 생성 ===
        # c_t = sum_i (alpha_{t,i} * h_i)
        # attention_weights: (T_x,) → (T_x, 1)로 확장하여 h_i와 곱합
        context = np.zeros(self.d_h)  # (d_h,) 크기의 빈 벡터
        for i in range(T_x):  # 각 Encoder 시점에 대해 반복
            # alpha_i * h_i를 누적합산
            context += attention_weights[i] * encoder_outputs[i]
        
        return context, attention_weights  # 컨텍스트 벡터와 가중치 반환


# === Additive Attention 동작 테스트 ===
print("=" * 60)  # 구분선 출력
print("Additive Attention (Bahdanau) 테스트")  # 테스트 시작 안내
print("=" * 60)  # 구분선 출력

# 하이퍼파라미터 설정
d_s = 8   # Decoder 은닉 차원
d_h = 8   # Encoder 은닉 차원
d_a = 4   # Attention 은닉 차원
T_x = 5   # 입력 시퀀스 길이 (Encoder 시점 수)

# Attention 객체 생성
additive_attn = AdditiveAttention(d_s=d_s, d_h=d_h, d_a=d_a)

# 가짜 데이터 생성 (실제로는 RNN에서 나오는 값)
s_prev = np.random.randn(d_s)              # Decoder 이전 은닉 상태 (d_s,)
encoder_outputs = np.random.randn(T_x, d_h)  # Encoder 모든 은닉 상태 (T_x, d_h)

# Attention 순전파 실행
context, attn_weights = additive_attn.forward(s_prev, encoder_outputs)

# 결과 출력
print(f"\nDecoder 이전 은닉 상태 크기: {s_prev.shape}")         # (d_s,) 확인
print(f"Encoder 출력 크기: {encoder_outputs.shape}")            # (T_x, d_h) 확인
print(f"\nAttention 가중치: {attn_weights}")                    # 각 시점의 가중치
print(f"Attention 가중치 합: {attn_weights.sum():.6f}")          # 합이 1인지 확인
print(f"\n컨텍스트 벡터 크기: {context.shape}")                  # (d_h,) 확인
print(f"컨텍스트 벡터: {context}")                               # 실제 값 출력

## 2.3 Dot-Product Attention (내적 어텐션) 클래스 구현

Luong의 Dot-Product Attention을 구현합니다. Additive보다 훨씬 간단합니다!

$$e_{t,i} = s_t^T h_i$$
$$\alpha_{t,i} = \text{softmax}(e_{t,i})$$
$$c_t = \sum_i \alpha_{t,i} h_i$$

쉽게 말하면, 두 벡터를 **내적(dot product)**하면 유사할수록 큰 값이 나옵니다.
이것이 곧 "관련도 점수"가 됩니다. 학습할 가중치가 없어서 가장 빠르고 단순합니다!

In [ ]:
class DotProductAttention:
    """Luong의 Dot-Product Attention을 NumPy로 구현한 클래스입니다.
    
    수식: e_{t,i} = s_t^T * h_i (단순 내적)
    
    전제 조건: Decoder와 Encoder의 은닉 차원이 같아야 합니다 (d_s = d_h)
    """
    
    def __init__(self, scaled=False):
        """Dot-Product Attention을 초기화합니다.
        
        Args:
            scaled: True이면 Scaled Dot-Product Attention 사용
                    (Transformer에서 사용하는 방식)
        """
        # 학습할 파라미터가 없습니다! (Additive와의 큰 차이)
        self.scaled = scaled  # 스케일링 여부 저장
    
    def forward(self, s_t, encoder_outputs):
        """Dot-Product Attention의 전체 순전파를 수행합니다.
        
        Args:
            s_t: Decoder의 현재 시점 은닉 상태 (d,)
                 (Luong은 현재 시점 s_t를 사용합니다!)
            encoder_outputs: Encoder의 모든 은닉 상태 (T_x, d)
        
        Returns:
            context: 컨텍스트 벡터 (d,)
            attention_weights: attention 가중치 (T_x,)
        """
        # 입력 시퀀스 길이와 은닉 차원을 가져옵니다
        T_x = encoder_outputs.shape[0]  # Encoder 시점 수
        d = encoder_outputs.shape[1]     # 은닉 차원 크기
        
        # === Step 1: 모든 Encoder 시점에 대한 score 계산 (내적) ===
        # s_t^T @ encoder_outputs^T = (d,) @ (d, T_x) 대신
        # encoder_outputs @ s_t = (T_x, d) @ (d,) = (T_x,) 로 한 번에 계산
        scores = encoder_outputs @ s_t  # (T_x,) - 각 시점의 내적 점수
        
        # Scaled Dot-Product인 경우 sqrt(d)로 나눕니다
        if self.scaled:  # 스케일링 옵션이 켜져 있으면
            scores = scores / np.sqrt(d)  # 분산을 1로 정규화
        
        # === Step 2: softmax로 정렬 가중치 계산 ===
        attention_weights = softmax(scores)  # (T_x,) - 확률 분포로 변환
        
        # === Step 3: 가중합으로 컨텍스트 벡터 생성 ===
        # attention_weights: (T_x,), encoder_outputs: (T_x, d)
        # (T_x,) @ (T_x, d) = (d,) - 행렬 곱으로 한 번에 가중합!
        context = attention_weights @ encoder_outputs  # (d,)
        
        return context, attention_weights  # 컨텍스트 벡터와 가중치 반환


# === Dot-Product Attention 동작 테스트 ===
print("=" * 60)  # 구분선 출력
print("Dot-Product Attention (Luong) 테스트")  # 테스트 시작 안내
print("=" * 60)  # 구분선 출력

# 하이퍼파라미터 설정 (d_s = d_h 필수!)
d = 8    # 공통 은닉 차원 (Encoder와 Decoder가 같아야 함)
T_x = 5  # 입력 시퀀스 길이

# Attention 객체 생성
dot_attn = DotProductAttention(scaled=False)           # 일반 dot-product
scaled_dot_attn = DotProductAttention(scaled=True)     # scaled dot-product

# 가짜 데이터 생성
s_t = np.random.randn(d)                    # Decoder 현재 은닉 상태 (d,)
enc_out = np.random.randn(T_x, d)           # Encoder 모든 은닉 상태 (T_x, d)

# 일반 Dot-Product Attention 실행
ctx1, w1 = dot_attn.forward(s_t, enc_out)

# Scaled Dot-Product Attention 실행
ctx2, w2 = scaled_dot_attn.forward(s_t, enc_out)

# 결과 비교 출력
print(f"\n일반 Dot-Product 가중치:  {w1}")       # 일반 버전의 가중치
print(f"Scaled Dot-Product 가중치: {w2}")        # 스케일링 버전의 가중치
print(f"\n일반 가중치 합: {w1.sum():.6f}")         # 합이 1인지 확인
print(f"Scaled 가중치 합: {w2.sum():.6f}")        # 합이 1인지 확인
print(f"\n차이점: Scaled 버전은 가중치가 더 균등합니다 (분산이 작음)")  # 설명
print(f"일반 가중치 분산: {w1.var():.6f}")         # 일반 버전 분산
print(f"Scaled 가중치 분산: {w2.var():.6f}")       # 스케일링 버전 분산 (더 작음)

## 2.4 작은 번역 예제로 Attention 가중치 직접 계산

간단한 한국어 → 영어 번역 예제에서 Attention이 실제로 어떻게 동작하는지 직접 계산해봅시다.

**예제 문장**: "나는 학생 입니다" → "I am a student"

아래에서는 실제 RNN/LSTM 대신, **우리가 직접 의미를 반영한 가상의 은닉 상태 벡터**를 만들어 Attention을 시뮬레이션합니다.
예를 들어 "나는"의 벡터와 "I"의 벡터가 비슷하도록 설정하면, Attention이 자연스럽게 "I" 생성 시 "나는"에 주목하게 됩니다.

In [ ]:
# ============================================================
# 작은 번역 예제: "나는 학생 입니다" → "I am a student"
# 실제 Encoder/Decoder가 만들 법한 은닉 상태를 시뮬레이션합니다.
# ============================================================

# --- 설정 ---
src_words = ["나는", "학생", "입니다"]  # 소스(한국어) 단어 리스트
tgt_words = ["I", "am", "a", "student"]  # 타겟(영어) 단어 리스트
d = 6  # 은닉 차원 (작은 예제이므로 6차원)

# --- 가상의 Encoder 은닉 상태 ---
# 실제로는 RNN/LSTM이 만들어내는 값이지만, 여기서는 직접 설정합니다.
# 각 한국어 단어가 특정 "의미 벡터"를 가진다고 상상합니다.
np.random.seed(42)  # 재현성을 위한 시드 고정

# Encoder 은닉 상태: (3, 6) - 3개 단어, 각 6차원
encoder_states = np.array([
    [0.8, 0.1, -0.2, 0.5, -0.1, 0.3],   # "나는" → 주어 관련 정보가 강함
    [-0.1, 0.9, 0.7, -0.2, 0.4, -0.1],   # "학생" → 명사/직업 관련 정보가 강함
    [0.2, -0.1, 0.1, 0.8, 0.6, 0.7],     # "입니다" → 동사/서술 관련 정보가 강함
])

# --- 가상의 Decoder 은닉 상태 ---
# 각 영어 단어를 생성하기 직전의 Decoder 상태
decoder_states = np.array([
    [0.7, 0.2, -0.1, 0.4, -0.2, 0.2],   # "I"를 생성하려는 상태 → "나는"과 유사
    [0.1, -0.1, 0.1, 0.7, 0.5, 0.6],    # "am"을 생성하려는 상태 → "입니다"와 유사
    [0.0, 0.1, 0.2, 0.6, 0.4, 0.5],     # "a"를 생성하려는 상태 → "입니다"와 유사
    [-0.1, 0.8, 0.6, -0.1, 0.3, -0.1],  # "student"를 생성하려는 상태 → "학생"과 유사
])

# --- Dot-Product Attention으로 가중치 계산 ---
# 모든 Decoder 시점에서의 Attention 가중치를 한 번에 계산합니다
dot_attn = DotProductAttention(scaled=True)  # Scaled Dot-Product 사용

# 결과를 저장할 행렬: (타겟 단어 수, 소스 단어 수) = (4, 3)
all_weights = np.zeros((len(tgt_words), len(src_words)))

# 각 Decoder 시점별로 Attention 계산
for t in range(len(tgt_words)):  # 각 타겟 단어에 대해
    # t번째 Decoder 상태로 Attention 수행
    context, weights = dot_attn.forward(
        decoder_states[t],  # 현재 Decoder 은닉 상태
        encoder_states       # 모든 Encoder 은닉 상태
    )
    all_weights[t] = weights  # 가중치 저장
    
    # 결과 출력: 어떤 소스 단어에 주목하는지
    print(f"\n'{tgt_words[t]}' 생성 시 Attention 가중치:")  # 타겟 단어 출력
    for i, (src_w, w) in enumerate(zip(src_words, weights)):  # 소스 단어별 가중치
        # 가중치 크기에 따라 막대 그래프 표시
        bar = "█" * int(w * 30)  # 가중치에 비례하는 막대
        print(f"  {src_w}: {w:.4f} {bar}")  # 단어와 가중치, 막대 출력

# 전체 Attention 가중치 행렬 출력
print(f"\n전체 Attention 가중치 행렬 (행: 타겟, 열: 소스):")
print(f"{'':>10}", end="")  # 공백 패딩
for w in src_words:  # 열 헤더 (소스 단어)
    print(f"{w:>10}", end="")  # 소스 단어 출력
print()  # 줄바꿈
for t, tgt_w in enumerate(tgt_words):  # 각 행 (타겟 단어)
    print(f"{tgt_w:>10}", end="")  # 타겟 단어 출력
    for i in range(len(src_words)):  # 각 열 (소스 단어)
        print(f"{all_weights[t, i]:>10.4f}", end="")  # 가중치 출력
    print()  # 줄바꿈

## 2.5 Attention 가중치 히트맵(Heatmap) 시각화

Attention 가중치를 히트맵으로 시각화하면, 모델이 **어디를 보는지** 직관적으로 이해할 수 있습니다.

히트맵은 **색의 진하기로 값의 크기를 표현**하는 그래프입니다.
빨간색이 진할수록 해당 위치에 더 많은 주의를 기울인다는 의미입니다.

In [ ]:
def plot_attention_heatmap(weights, src_labels, tgt_labels, title="Attention 가중치 히트맵"):
    """Attention 가중치를 히트맵으로 시각화하는 함수입니다.
    
    Args:
        weights: Attention 가중치 행렬 (타겟 길이, 소스 길이)
        src_labels: 소스 단어 리스트 (x축 라벨)
        tgt_labels: 타겟 단어 리스트 (y축 라벨)
        title: 그래프 제목 (기본값: "Attention 가중치 히트맵")
    """
    # 그래프 크기 설정 (가로 8, 세로 6 인치)
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # seaborn 히트맵 그리기
    sns.heatmap(
        weights,              # Attention 가중치 행렬
        xticklabels=src_labels,  # x축: 소스(한국어) 단어
        yticklabels=tgt_labels,  # y축: 타겟(영어) 단어
        annot=True,           # 각 셀에 숫자 표시
        fmt=".3f",            # 소수점 3자리까지 표시
        cmap="YlOrRd",        # 색상 맵: 노란색 → 주황색 → 빨간색
        vmin=0,               # 최소값 0
        vmax=1,               # 최대값 1
        linewidths=0.5,       # 셀 사이 구분선 두께
        ax=ax                 # 그릴 축 지정
    )
    
    # 제목과 축 라벨 설정
    ax.set_title(title, fontsize=14, pad=15)      # 그래프 제목
    ax.set_xlabel("소스 (한국어)", fontsize=12)      # x축 라벨
    ax.set_ylabel("타겟 (영어)", fontsize=12)        # y축 라벨
    
    # 레이아웃 자동 조정 (라벨이 잘리지 않도록)
    plt.tight_layout()
    
    # 그래프 표시
    plt.show()


# === 번역 예제의 Attention 히트맵 시각화 ===
plot_attention_heatmap(
    weights=all_weights,        # 위에서 계산한 Attention 가중치 행렬
    src_labels=src_words,       # 소스 단어: ["나는", "학생", "입니다"]
    tgt_labels=tgt_words,       # 타겟 단어: ["I", "am", "a", "student"]
    title="한→영 번역 Attention 가중치 (Dot-Product)"  # 그래프 제목
)

In [ ]:
# ============================================================
# Additive Attention으로도 같은 예제를 실행해봅시다.
# Dot-Product과 결과가 어떻게 다른지 비교합니다.
# ============================================================

# Additive Attention 객체 생성 (d_s=6, d_h=6, d_a=4)
np.random.seed(123)  # 재현성을 위한 시드 고정
add_attn = AdditiveAttention(d_s=6, d_h=6, d_a=4)  # Additive Attention 생성

# 결과를 저장할 행렬: (4, 3) - 타겟 4개, 소스 3개
add_weights = np.zeros((len(tgt_words), len(src_words)))

# 각 Decoder 시점별로 Attention 계산
for t in range(len(tgt_words)):  # 각 타겟 단어에 대해
    # Additive Attention은 s_{t-1} (이전 시점)을 사용하지만,
    # 여기서는 같은 decoder_states를 사용하여 비교합니다
    context, weights = add_attn.forward(
        decoder_states[t],  # Decoder 은닉 상태
        encoder_states       # Encoder 모든 은닉 상태
    )
    add_weights[t] = weights  # 가중치 저장

# Additive Attention 히트맵 시각화
plot_attention_heatmap(
    weights=add_weights,        # Additive Attention 가중치
    src_labels=src_words,       # 소스 단어
    tgt_labels=tgt_words,       # 타겟 단어
    title="한→영 번역 Attention 가중치 (Additive/Bahdanau)"  # 제목
)

# 두 방식의 차이 설명
print("Dot-Product vs Additive 가중치 차이:")  # 제목 출력
for t, tgt_w in enumerate(tgt_words):  # 각 타겟 단어별
    print(f"\n  '{tgt_w}' 생성 시:")  # 타겟 단어 출력
    for i, src_w in enumerate(src_words):  # 각 소스 단어별
        # 두 방식의 가중치를 나란히 비교
        diff = abs(all_weights[t, i] - add_weights[t, i])  # 가중치 차이
        print(f"    {src_w}: Dot={all_weights[t,i]:.3f}  Add={add_weights[t,i]:.3f}  차이={diff:.3f}")

---

# Part 3: PyTorch로 Seq2Seq + Attention 구현

> **이 파트에서 배울 것**: PyTorch로 실제 동작하는 Seq2Seq + Bahdanau Attention 모델을 구현합니다.
> 또한 Attention을 감정 분류에 적용하는 예제도 다룹니다.

---

## 3.1 Encoder: Bidirectional(양방향) LSTM

Bahdanau 논문에서는 **양방향(Bidirectional) RNN**을 Encoder로 사용합니다.

양방향이란? 문장을 **앞→뒤**, **뒤→앞** 두 방향으로 읽어서,
각 시점 $i$에서 **앞뒤 문맥을 모두** 담은 은닉 상태를 만듭니다:

$$h_i = [\overrightarrow{h_i} ; \overleftarrow{h_i}]$$

- $\overrightarrow{h_i}$: 앞→뒤 방향에서 $i$번째 시점의 은닉 상태
- $\overleftarrow{h_i}$: 뒤→앞 방향에서 $i$번째 시점의 은닉 상태
- 따라서 $h_i$의 차원은 $2 \times d_h$ (두 방향을 연결)

쉽게 말하면, 영어 문장을 **앞에서부터도 읽고, 뒤에서부터도 읽어서** 각 단어의 의미를 더 풍부하게 파악하는 것입니다.
예: "나는 사과를 좋아한다"에서 "사과"의 의미를 파악할 때, 앞의 "나는"도, 뒤의 "좋아한다"도 참고합니다.

In [ ]:
class Encoder(nn.Module):
    """양방향(Bidirectional) LSTM Encoder입니다.
    
    입력 시퀀스를 받아 각 시점의 은닉 상태를 반환합니다.
    양방향이므로 각 시점의 은닉 상태 차원은 hidden_dim * 2 입니다.
    
    Args:
        vocab_size: 어휘 크기 (정수)
        embed_dim: 임베딩 차원 (정수)
        hidden_dim: LSTM 은닉 차원 (정수, 양방향이므로 실제 출력은 2배)
        n_layers: LSTM 층 수 (정수, 기본값 1)
        dropout: 드롭아웃 비율 (실수, 기본값 0.1)
    """
    
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers=1, dropout=0.1):
        """Encoder를 초기화합니다."""
        super().__init__()  # nn.Module 초기화 (필수)
        
        # 임베딩 층: 단어 인덱스 → 밀집 벡터로 변환
        # vocab_size개의 단어를 embed_dim 차원 벡터로 매핑
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # 양방향 LSTM: 시퀀스를 앞→뒤, 뒤→앞 두 방향으로 읽음
        self.lstm = nn.LSTM(
            input_size=embed_dim,       # 입력 차원 (임베딩 차원)
            hidden_size=hidden_dim,     # 은닉 차원 (한 방향 기준)
            num_layers=n_layers,        # LSTM 층 수
            batch_first=True,           # 입력 형태: (배치, 시퀀스, 특성)
            bidirectional=True,         # 양방향 활성화!
            dropout=dropout if n_layers > 1 else 0  # 다층일 때만 드롭아웃
        )
        
        # Decoder 초기 상태 생성용 선형 변환
        # 양방향 LSTM의 마지막 상태(2*hidden_dim)를 Decoder 차원(hidden_dim)으로 변환
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)  # h 변환
        self.fc_cell = nn.Linear(hidden_dim * 2, hidden_dim)    # c 변환
        
        # 드롭아웃 층: 과적합 방지
        self.dropout = nn.Dropout(dropout)
        
        # 차원 정보 저장 (디버깅용)
        self.hidden_dim = hidden_dim  # 은닉 차원
        self.n_layers = n_layers      # LSTM 층 수
    
    def forward(self, src):
        """Encoder 순전파를 수행합니다.
        
        Args:
            src: 입력 시퀀스 (batch_size, src_len) - 단어 인덱스
        
        Returns:
            encoder_outputs: 모든 시점의 은닉 상태 (batch_size, src_len, hidden_dim*2)
            hidden: Decoder 초기 은닉 상태 (1, batch_size, hidden_dim)
            cell: Decoder 초기 셀 상태 (1, batch_size, hidden_dim)
        """
        # Step 1: 단어 인덱스를 임베딩 벡터로 변환
        # (batch_size, src_len) → (batch_size, src_len, embed_dim)
        embedded = self.dropout(self.embedding(src))
        
        # Step 2: 양방향 LSTM 통과
        # encoder_outputs: (batch_size, src_len, hidden_dim * 2)
        # hidden: (n_layers * 2, batch_size, hidden_dim) - 2는 양방향
        # cell: (n_layers * 2, batch_size, hidden_dim)
        encoder_outputs, (hidden, cell) = self.lstm(embedded)
        
        # Step 3: Decoder 초기 상태 생성
        # 양방향 LSTM의 마지막 층에서 앞→뒤와 뒤→앞 방향의 상태를 연결
        # hidden[-2]: 마지막 층의 앞→뒤 방향 상태 (batch_size, hidden_dim)
        # hidden[-1]: 마지막 층의 뒤→앞 방향 상태 (batch_size, hidden_dim)
        hidden_cat = torch.cat((hidden[-2], hidden[-1]), dim=1)  # (batch, hidden*2)
        cell_cat = torch.cat((cell[-2], cell[-1]), dim=1)        # (batch, hidden*2)
        
        # 선형 변환 + tanh로 Decoder 차원에 맞춤
        # (batch, hidden*2) → (batch, hidden)
        hidden = torch.tanh(self.fc_hidden(hidden_cat)).unsqueeze(0)  # (1, batch, hidden)
        cell = torch.tanh(self.fc_cell(cell_cat)).unsqueeze(0)        # (1, batch, hidden)
        
        return encoder_outputs, hidden, cell  # Encoder 출력과 초기 상태 반환


# === Encoder 동작 테스트 ===
print("=" * 60)  # 구분선
print("Encoder (Bidirectional LSTM) 테스트")  # 제목
print("=" * 60)  # 구분선

# 하이퍼파라미터 설정
VOCAB_SIZE = 100    # 어휘 크기 (단어 100개)
EMBED_DIM = 16      # 임베딩 차원
HIDDEN_DIM = 32     # LSTM 은닉 차원 (양방향이므로 출력은 64)
BATCH_SIZE = 2      # 배치 크기 (문장 2개)
SRC_LEN = 5         # 입력 시퀀스 길이 (단어 5개)

# Encoder 생성
encoder = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)

# 가짜 입력 데이터: (배치 2, 길이 5) 크기의 단어 인덱스
fake_src = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SRC_LEN))  # 랜덤 단어 인덱스

# Encoder 순전파 실행
enc_out, enc_hidden, enc_cell = encoder(fake_src)

# 결과 크기 확인
print(f"\n입력 크기: {fake_src.shape}")                    # (2, 5)
print(f"Encoder 출력 크기: {enc_out.shape}")               # (2, 5, 64) - hidden*2
print(f"Decoder 초기 hidden 크기: {enc_hidden.shape}")     # (1, 2, 32)
print(f"Decoder 초기 cell 크기: {enc_cell.shape}")         # (1, 2, 32)
print(f"\n양방향이므로 Encoder 출력의 마지막 차원 = {HIDDEN_DIM} * 2 = {HIDDEN_DIM*2}")  # 설명

## 3.2 Bahdanau Attention 모듈 (PyTorch)

NumPy로 구현한 Additive Attention을 PyTorch `nn.Module`로 옮깁니다.

> **NumPy 구현과의 차이점**: PyTorch 버전은 **배치(batch) 처리**를 지원합니다.
> 즉, 여러 문장을 한 번에 처리할 수 있어서 GPU 활용이 가능합니다.
> 또한 `nn.Linear`를 사용하면 가중치를 자동으로 초기화하고 기울기를 추적합니다.

In [ ]:
class BahdanauAttention(nn.Module):
    """PyTorch로 구현한 Bahdanau (Additive) Attention 모듈입니다.
    
    수식: e_{t,i} = v^T * tanh(W_s * s_{t-1} + W_h * h_i)
    
    Args:
        enc_hidden_dim: Encoder 은닉 차원 (양방향이면 원래 차원 * 2)
        dec_hidden_dim: Decoder 은닉 차원
        attn_dim: Attention 은닉 차원 (하이퍼파라미터)
    """
    
    def __init__(self, enc_hidden_dim, dec_hidden_dim, attn_dim):
        """Attention 가중치 행렬들을 초기화합니다."""
        super().__init__()  # nn.Module 초기화 (필수)
        
        # W_s: Decoder 상태를 attention 공간으로 변환
        # (dec_hidden_dim) → (attn_dim)
        self.W_s = nn.Linear(dec_hidden_dim, attn_dim, bias=False)
        
        # W_h: Encoder 상태를 attention 공간으로 변환
        # (enc_hidden_dim) → (attn_dim)
        self.W_h = nn.Linear(enc_hidden_dim, attn_dim, bias=False)
        
        # v: attention 공간의 벡터를 스칼라 점수로 변환
        # (attn_dim) → (1)
        self.v = nn.Linear(attn_dim, 1, bias=False)
    
    def forward(self, decoder_hidden, encoder_outputs):
        """Bahdanau Attention의 순전파를 수행합니다.
        
        Args:
            decoder_hidden: Decoder의 이전 은닉 상태 (batch_size, dec_hidden_dim)
            encoder_outputs: Encoder의 모든 은닉 상태 (batch_size, src_len, enc_hidden_dim)
        
        Returns:
            context: 컨텍스트 벡터 (batch_size, enc_hidden_dim)
            attention_weights: Attention 가중치 (batch_size, src_len)
        """
        # 입력 시퀀스 길이를 가져옵니다
        src_len = encoder_outputs.shape[1]  # 소스 시퀀스 길이
        
        # === Step 1: Decoder 상태를 attention 공간으로 변환 ===
        # decoder_hidden: (batch, dec_hidden) → (batch, attn_dim)
        # 그런데 encoder_outputs은 (batch, src_len, ...) 이므로
        # decoder_hidden도 src_len 차원을 추가해야 더할 수 있습니다
        # (batch, dec_hidden) → (batch, 1, dec_hidden) → repeat → (batch, src_len, dec_hidden)
        decoder_hidden_expanded = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)
        
        # W_s * s_{t-1}: (batch, src_len, dec_hidden) → (batch, src_len, attn_dim)
        proj_s = self.W_s(decoder_hidden_expanded)
        
        # === Step 2: Encoder 상태를 attention 공간으로 변환 ===
        # W_h * h_i: (batch, src_len, enc_hidden) → (batch, src_len, attn_dim)
        proj_h = self.W_h(encoder_outputs)
        
        # === Step 3: Additive Score 계산 ===
        # tanh(W_s * s + W_h * h): (batch, src_len, attn_dim)
        energy = torch.tanh(proj_s + proj_h)
        
        # v^T * tanh(...): (batch, src_len, attn_dim) → (batch, src_len, 1) → (batch, src_len)
        scores = self.v(energy).squeeze(-1)  # 마지막 차원(1) 제거
        
        # === Step 4: softmax로 정렬 가중치 계산 ===
        # (batch, src_len) → (batch, src_len) - 각 행이 확률 분포
        attention_weights = torch.softmax(scores, dim=-1)
        
        # === Step 5: 가중합으로 컨텍스트 벡터 생성 ===
        # attention_weights: (batch, src_len) → (batch, 1, src_len)
        # encoder_outputs: (batch, src_len, enc_hidden)
        # bmm (batch matrix multiply): (batch, 1, src_len) @ (batch, src_len, enc_hidden)
        #                             = (batch, 1, enc_hidden) → (batch, enc_hidden)
        context = torch.bmm(
            attention_weights.unsqueeze(1),  # (batch, 1, src_len)
            encoder_outputs                   # (batch, src_len, enc_hidden)
        ).squeeze(1)  # (batch, enc_hidden) - 1차원 제거
        
        return context, attention_weights  # 컨텍스트 벡터와 가중치 반환


# === Bahdanau Attention 모듈 테스트 ===
print("=" * 60)  # 구분선
print("Bahdanau Attention (PyTorch) 테스트")  # 제목
print("=" * 60)  # 구분선

# 하이퍼파라미터 설정
ENC_HIDDEN = 64     # Encoder 은닉 차원 (양방향 32*2)
DEC_HIDDEN = 32     # Decoder 은닉 차원
ATTN_DIM = 16       # Attention 은닉 차원

# Attention 모듈 생성
attention = BahdanauAttention(ENC_HIDDEN, DEC_HIDDEN, ATTN_DIM)

# 가짜 데이터 생성
fake_dec_hidden = torch.randn(BATCH_SIZE, DEC_HIDDEN)       # Decoder 은닉 상태
fake_enc_outputs = torch.randn(BATCH_SIZE, SRC_LEN, ENC_HIDDEN)  # Encoder 출력

# Attention 순전파 실행
ctx, attn_w = attention(fake_dec_hidden, fake_enc_outputs)

# 결과 크기 확인
print(f"\nDecoder 은닉 상태 크기: {fake_dec_hidden.shape}")    # (2, 32)
print(f"Encoder 출력 크기: {fake_enc_outputs.shape}")          # (2, 5, 64)
print(f"컨텍스트 벡터 크기: {ctx.shape}")                      # (2, 64)
print(f"Attention 가중치 크기: {attn_w.shape}")                # (2, 5)
print(f"\nAttention 가중치 (첫 번째 배치): {attn_w[0].detach().numpy()}")  # 가중치 값
print(f"가중치 합: {attn_w[0].sum().item():.6f}")              # 합이 1인지 확인

## 3.3 Decoder with Bahdanau Attention

이제 Attention을 내장한 Decoder를 구현합니다.
Decoder는 매 시점마다 Attention을 통해 Encoder의 어느 부분을 볼지 결정합니다.

```
매 시점(time step)마다 Decoder가 하는 일:
┌──────────────────────────────────────────────┐
│ 1. 이전 은닉 상태(s_{t-1})로 Attention 수행    │
│    → "입력 문장의 어디를 볼까?" 결정            │
│ 2. 컨텍스트 벡터(c_t) 생성                     │
│    → 관련 입력 정보를 가중합                    │
│ 3. [이전 출력 임베딩 ; c_t]를 LSTM에 입력       │
│    → 새로운 은닉 상태 생성                      │
│ 4. 출력 단어 예측                              │
│    → [은닉 상태 ; c_t ; 임베딩]으로 최종 예측   │
└──────────────────────────────────────────────┘
```

In [ ]:
class DecoderWithAttention(nn.Module):
    """Bahdanau Attention을 내장한 LSTM Decoder입니다.
    
    매 시점마다:
    1. 이전 은닉 상태로 Attention 가중치 계산
    2. 컨텍스트 벡터 생성
    3. 컨텍스트 + 임베딩을 LSTM에 입력
    4. 출력 단어 예측
    
    Args:
        vocab_size: 출력 어휘 크기 (정수)
        embed_dim: 임베딩 차원 (정수)
        enc_hidden_dim: Encoder 은닉 차원 (양방향이면 * 2)
        dec_hidden_dim: Decoder 은닉 차원 (정수)
        attn_dim: Attention 은닉 차원 (정수)
        dropout: 드롭아웃 비율 (실수, 기본값 0.1)
    """
    
    def __init__(self, vocab_size, embed_dim, enc_hidden_dim, dec_hidden_dim, attn_dim, dropout=0.1):
        """Decoder를 초기화합니다."""
        super().__init__()  # nn.Module 초기화
        
        # Attention 모듈 (위에서 구현한 BahdanauAttention)
        self.attention = BahdanauAttention(enc_hidden_dim, dec_hidden_dim, attn_dim)
        
        # 임베딩 층: 출력 단어 인덱스 → 벡터
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # LSTM: 입력 = [임베딩 ; 컨텍스트 벡터]를 연결한 것
        # 따라서 입력 차원 = embed_dim + enc_hidden_dim
        self.lstm = nn.LSTM(
            input_size=embed_dim + enc_hidden_dim,  # 임베딩 + 컨텍스트
            hidden_size=dec_hidden_dim,              # Decoder 은닉 차원
            num_layers=1,                            # 1층
            batch_first=True                         # (배치, 시퀀스, 특성)
        )
        
        # 출력층: [Decoder 은닉 ; 컨텍스트 ; 임베딩]을 연결하여 단어 예측
        # 이렇게 3가지를 연결하는 것은 Bahdanau 논문의 deep output 기법입니다
        self.fc_out = nn.Linear(
            dec_hidden_dim + enc_hidden_dim + embed_dim,  # 3가지 연결
            vocab_size                                     # 어휘 크기만큼 출력
        )
        
        # 드롭아웃: 과적합 방지
        self.dropout = nn.Dropout(dropout)
        
        # 차원 정보 저장
        self.vocab_size = vocab_size        # 어휘 크기
        self.enc_hidden_dim = enc_hidden_dim  # Encoder 은닉 차원
    
    def forward_step(self, input_token, hidden, cell, encoder_outputs):
        """Decoder의 한 시점(1 step) 순전파를 수행합니다.
        
        Args:
            input_token: 현재 시점 입력 토큰 (batch_size,) - 단어 인덱스
            hidden: 이전 시점 은닉 상태 (1, batch_size, dec_hidden_dim)
            cell: 이전 시점 셀 상태 (1, batch_size, dec_hidden_dim)
            encoder_outputs: Encoder 전체 출력 (batch_size, src_len, enc_hidden_dim)
        
        Returns:
            prediction: 단어 예측 로짓 (batch_size, vocab_size)
            hidden: 현재 시점 은닉 상태 (1, batch_size, dec_hidden_dim)
            cell: 현재 시점 셀 상태 (1, batch_size, dec_hidden_dim)
            attention_weights: Attention 가중치 (batch_size, src_len)
        """
        # === Step 1: Attention으로 컨텍스트 벡터 생성 ===
        # hidden[0]: (batch, dec_hidden) - squeeze로 1차원 제거
        context, attention_weights = self.attention(
            hidden.squeeze(0),   # (1, batch, dec_hidden) → (batch, dec_hidden)
            encoder_outputs       # (batch, src_len, enc_hidden)
        )
        # context: (batch, enc_hidden)
        # attention_weights: (batch, src_len)
        
        # === Step 2: 입력 토큰을 임베딩 ===
        # (batch,) → (batch, embed_dim)
        embedded = self.dropout(self.embedding(input_token))
        
        # === Step 3: [임베딩 ; 컨텍스트]를 연결하여 LSTM 입력 생성 ===
        # (batch, embed_dim) + (batch, enc_hidden) → (batch, embed_dim + enc_hidden)
        lstm_input = torch.cat([embedded, context], dim=1)
        
        # LSTM은 (batch, seq_len, features) 형태를 기대하므로 seq_len=1 차원 추가
        lstm_input = lstm_input.unsqueeze(1)  # (batch, 1, embed_dim + enc_hidden)
        
        # === Step 4: LSTM 통과 ===
        # lstm_output: (batch, 1, dec_hidden)
        # hidden, cell: (1, batch, dec_hidden)
        lstm_output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        
        # lstm_output에서 seq_len 차원 제거: (batch, dec_hidden)
        lstm_output = lstm_output.squeeze(1)
        
        # === Step 5: Deep Output으로 단어 예측 ===
        # [Decoder 출력 ; 컨텍스트 ; 임베딩]을 연결
        combined = torch.cat([lstm_output, context, embedded], dim=1)
        # (batch, dec_hidden + enc_hidden + embed_dim) → (batch, vocab_size)
        prediction = self.fc_out(combined)
        
        return prediction, hidden, cell, attention_weights


# === Decoder with Attention 테스트 ===
print("=" * 60)  # 구분선
print("Decoder with Bahdanau Attention 테스트")  # 제목
print("=" * 60)  # 구분선

# 하이퍼파라미터
TGT_VOCAB_SIZE = 80  # 타겟 어휘 크기
TGT_EMBED_DIM = 16   # 타겟 임베딩 차원

# Decoder 생성
decoder = DecoderWithAttention(
    vocab_size=TGT_VOCAB_SIZE,      # 타겟 어휘 크기
    embed_dim=TGT_EMBED_DIM,        # 임베딩 차원
    enc_hidden_dim=HIDDEN_DIM * 2,  # Encoder 출력 차원 (양방향)
    dec_hidden_dim=HIDDEN_DIM,      # Decoder 은닉 차원
    attn_dim=ATTN_DIM               # Attention 은닉 차원
)

# 가짜 입력: 첫 번째 시점의 타겟 토큰 (보통 <SOS> 토큰)
fake_input = torch.randint(0, TGT_VOCAB_SIZE, (BATCH_SIZE,))  # (batch,)

# Decoder 한 시점 실행
pred, dec_h, dec_c, dec_attn = decoder.forward_step(
    fake_input,    # 현재 입력 토큰
    enc_hidden,    # Encoder에서 받은 초기 은닉 상태
    enc_cell,      # Encoder에서 받은 초기 셀 상태
    enc_out         # Encoder 전체 출력
)

# 결과 크기 확인
print(f"\n입력 토큰 크기: {fake_input.shape}")           # (2,)
print(f"예측 로짓 크기: {pred.shape}")                   # (2, 80) - 어휘 크기
print(f"새 은닉 상태 크기: {dec_h.shape}")                # (1, 2, 32)
print(f"Attention 가중치 크기: {dec_attn.shape}")         # (2, 5)
print(f"\nAttention 가중치: {dec_attn[0].detach().numpy()}")  # 첫 배치의 가중치
print(f"예측된 단어 인덱스: {pred.argmax(dim=-1).tolist()}")   # argmax로 단어 선택

## 3.4 감정 분류용 Attention (Classification Attention)

Attention은 번역뿐 아니라 **텍스트 분류**에도 매우 유용합니다.

문장을 LSTM으로 읽은 후, "어떤 단어가 감정 판단에 가장 중요한지"를
Attention으로 학습합니다. 이를 통해:
1. 더 좋은 문장 표현(representation)을 얻을 수 있고
2. **해석 가능성(interpretability, 모델이 왜 그런 결정을 했는지 설명 가능)** 을 확보할 수 있습니다

쉽게 말하면, "이 영화는 정말 **재미있고** **감동적인** 작품이다"라는 문장에서
감정 분류 모델이 "재미있고"와 "감동적인"에 높은 Attention을 주면,
사람도 "아, 모델이 이 단어들 때문에 긍정으로 판단했구나"라고 이해할 수 있습니다.

In [ ]:
class SentimentAttentionClassifier(nn.Module):
    """Attention 기반 감정 분류기입니다.
    
    구조:
    1. 임베딩 → BiLSTM → 각 시점의 은닉 상태 획득
    2. Attention으로 각 시점의 중요도(가중치) 계산
    3. 가중합으로 문장 벡터 생성
    4. 분류기(FC)로 감정 예측
    
    Args:
        vocab_size: 어휘 크기
        embed_dim: 임베딩 차원
        hidden_dim: LSTM 은닉 차원 (양방향이면 출력은 2배)
        num_classes: 분류 클래스 수 (예: 긍정/부정 = 2)
        dropout: 드롭아웃 비율
    """
    
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.1):
        """감정 분류기를 초기화합니다."""
        super().__init__()  # nn.Module 초기화
        
        # 임베딩 층: 단어 → 벡터
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # 양방향 LSTM: 문맥을 양방향으로 읽음
        self.lstm = nn.LSTM(
            input_size=embed_dim,     # 임베딩 차원
            hidden_size=hidden_dim,   # 은닉 차원
            batch_first=True,         # (배치, 시퀀스, 특성)
            bidirectional=True        # 양방향
        )
        
        # Self-Attention 가중치
        # LSTM 출력(hidden*2)을 스칼라 점수로 변환하는 2층 네트워크
        self.attention_fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),  # (hidden*2) → (hidden)
            nn.Tanh(),                               # tanh 활성화
            nn.Linear(hidden_dim, 1)                  # (hidden) → (1) 스칼라 점수
        )
        
        # 분류기: 문장 벡터 → 클래스 예측
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),                      # 드롭아웃
            nn.Linear(hidden_dim * 2, hidden_dim),    # 차원 축소
            nn.ReLU(),                                # ReLU 활성화
            nn.Dropout(dropout),                      # 드롭아웃
            nn.Linear(hidden_dim, num_classes)         # 최종 분류
        )
    
    def forward(self, text):
        """감정 분류의 순전파를 수행합니다.
        
        Args:
            text: 입력 텍스트 (batch_size, seq_len) - 단어 인덱스
        
        Returns:
            logits: 클래스 예측 로짓 (batch_size, num_classes)
            attention_weights: Attention 가중치 (batch_size, seq_len)
        """
        # === Step 1: 임베딩 ===
        # (batch, seq_len) → (batch, seq_len, embed_dim)
        embedded = self.embedding(text)
        
        # === Step 2: BiLSTM으로 문맥 인코딩 ===
        # lstm_out: (batch, seq_len, hidden*2) - 양방향이라 2배
        lstm_out, _ = self.lstm(embedded)
        
        # === Step 3: Attention 가중치 계산 ===
        # 각 시점의 LSTM 출력에 대해 "중요도 점수" 계산
        # (batch, seq_len, hidden*2) → (batch, seq_len, 1) → (batch, seq_len)
        attn_scores = self.attention_fc(lstm_out).squeeze(-1)
        
        # softmax로 확률 분포로 변환
        attention_weights = torch.softmax(attn_scores, dim=-1)  # (batch, seq_len)
        
        # === Step 4: 가중합으로 문장 벡터 생성 ===
        # attention_weights: (batch, seq_len) → (batch, 1, seq_len)
        # lstm_out: (batch, seq_len, hidden*2)
        # bmm: (batch, 1, seq_len) @ (batch, seq_len, hidden*2) = (batch, 1, hidden*2)
        sentence_vector = torch.bmm(
            attention_weights.unsqueeze(1),  # (batch, 1, seq_len)
            lstm_out                          # (batch, seq_len, hidden*2)
        ).squeeze(1)  # (batch, hidden*2)
        
        # === Step 5: 분류 ===
        # (batch, hidden*2) → (batch, num_classes)
        logits = self.classifier(sentence_vector)
        
        return logits, attention_weights


# === 감정 분류 모델 테스트 ===
print("=" * 60)  # 구분선
print("감정 분류 + Attention 테스트")  # 제목
print("=" * 60)  # 구분선

# 하이퍼파라미터
SENT_VOCAB = 200      # 어휘 크기
SENT_EMBED = 32       # 임베딩 차원
SENT_HIDDEN = 64      # LSTM 은닉 차원
NUM_CLASSES = 2       # 클래스 수 (긍정/부정)
SEQ_LEN = 8           # 시퀀스 길이

# 모델 생성
sentiment_model = SentimentAttentionClassifier(
    vocab_size=SENT_VOCAB,
    embed_dim=SENT_EMBED,
    hidden_dim=SENT_HIDDEN,
    num_classes=NUM_CLASSES
)

# 가짜 입력: (배치 4, 시퀀스 8)
fake_text = torch.randint(0, SENT_VOCAB, (4, SEQ_LEN))  # 4개 문장, 각 8단어

# 순전파 실행
logits, sent_attn = sentiment_model(fake_text)

# 결과 확인
print(f"\n입력 크기: {fake_text.shape}")                         # (4, 8)
print(f"출력 로짓 크기: {logits.shape}")                          # (4, 2)
print(f"Attention 가중치 크기: {sent_attn.shape}")                # (4, 8)
print(f"\n첫 문장의 Attention 가중치: {sent_attn[0].detach().numpy()}")  # 첫 문장 가중치
print(f"예측 클래스: {logits.argmax(dim=-1).tolist()}")             # 예측 결과

# 파라미터 수 출력
total_params = sum(p.numel() for p in sentiment_model.parameters())  # 전체 파라미터 수
print(f"\n전체 파라미터 수: {total_params:,}")  # 쉼표 포맷으로 출력

## 3.5 감정 분류 Attention 가중치 시각화

감정 분류에서 Attention이 **어떤 단어에 주목하는지** 시각화해봅시다.
가상의 감정 문장에 대해 Attention 가중치를 보면, 모델이 감정 판단의 근거를 이해할 수 있습니다.

> **참고**: 아래 예제는 학습되지 않은 모델이므로 가중치가 랜덤입니다.
> 실제 학습 후에는 "재미있고", "지루하고" 같은 감정 단어에 높은 가중치가 부여됩니다.
> 여기서는 **시각화 방법 자체를 이해하는 것**이 목적입니다.

In [ ]:
# ============================================================
# 감정 분류 Attention 시각화
# 가상의 감정 문장에 대해 Attention이 어떤 단어에 주목하는지 봅니다.
# (학습되지 않은 모델이므로 가중치는 랜덤이지만, 시각화 방법을 이해하는 것이 목적)
# ============================================================

def visualize_sentiment_attention(words, attention_weights, prediction, title=""):
    """감정 분류의 Attention 가중치를 단어 위에 색으로 시각화합니다.
    
    Args:
        words: 단어 리스트 (문자열 리스트)
        attention_weights: Attention 가중치 (단어 수 크기의 1D 배열)
        prediction: 예측 결과 문자열 ("긍정" 또는 "부정")
        title: 그래프 제목
    """
    # 그래프 생성 (가로로 긴 형태)
    fig, ax = plt.subplots(figsize=(12, 2))
    
    # 가중치를 numpy로 변환 (텐서일 수 있으므로)
    if hasattr(attention_weights, 'detach'):  # PyTorch 텐서인 경우
        weights = attention_weights.detach().numpy()
    else:  # 이미 numpy 배열인 경우
        weights = attention_weights
    
    # 각 단어를 색칠된 사각형으로 그립니다
    for i, (word, weight) in enumerate(zip(words, weights)):
        # 배경색: 가중치에 비례하여 빨간색 강도 조절
        color = plt.cm.Reds(weight / weights.max())  # 최대 가중치 대비 정규화
        
        # 사각형 그리기
        rect = plt.Rectangle(
            (i, 0),       # 좌하단 좌표
            0.95,          # 폭
            0.8,           # 높이
            facecolor=color,  # 배경색 (가중치에 비례)
            edgecolor='gray'  # 테두리색
        )
        ax.add_patch(rect)  # 사각형을 그래프에 추가
        
        # 단어 텍스트 표시
        ax.text(
            i + 0.475,    # x 좌표 (사각형 중앙)
            0.4,           # y 좌표
            word,          # 단어 텍스트
            ha='center',   # 수평 가운데 정렬
            va='center',   # 수직 가운데 정렬
            fontsize=11    # 폰트 크기
        )
        
        # 가중치 숫자 표시
        ax.text(
            i + 0.475,    # x 좌표
            0.1,           # y 좌표 (단어 아래)
            f"{weight:.3f}",  # 소수점 3자리
            ha='center',   # 수평 가운데
            va='center',   # 수직 가운데
            fontsize=8,    # 작은 폰트
            color='gray'   # 회색
        )
    
    # 축 설정
    ax.set_xlim(-0.1, len(words) + 0.1)   # x축 범위
    ax.set_ylim(-0.1, 1.0)                 # y축 범위
    ax.axis('off')                          # 축 숨기기
    ax.set_title(f"{title} (예측: {prediction})", fontsize=13, pad=10)  # 제목
    
    plt.tight_layout()  # 레이아웃 조정
    plt.show()          # 그래프 표시


# === 가상의 감정 분석 예제 ===
# 가상 어휘 사전 (단어 → 인덱스 매핑)
word_to_idx = {
    "<PAD>": 0, "이": 1, "영화": 2, "는": 3, "정말": 4,
    "재미있고": 5, "감동적인": 6, "작품": 7, "이다": 8,
    "너무": 9, "지루하고": 10, "실망스러운": 11,
    "최고의": 12, "최악의": 13, "훌륭한": 14, "별로인": 15
}

# 예제 문장들 (단어 리스트)
sentences = [
    ["이", "영화", "는", "정말", "재미있고", "감동적인", "작품", "이다"],  # 긍정
    ["이", "영화", "는", "너무", "지루하고", "실망스러운", "작품", "이다"],  # 부정
    ["정말", "최고의", "훌륭한", "작품", "이다", "<PAD>", "<PAD>", "<PAD>"],  # 긍정 (패딩)
]

# 단어를 인덱스로 변환
indexed_sentences = []  # 인덱스로 변환된 문장을 저장할 리스트
for sent in sentences:  # 각 문장에 대해
    # 각 단어를 인덱스로 변환 (없으면 0)
    indexed = [word_to_idx.get(w, 0) for w in sent]
    indexed_sentences.append(indexed)  # 리스트에 추가

# PyTorch 텐서로 변환
input_tensor = torch.tensor(indexed_sentences, dtype=torch.long)  # (3, 8) 정수 텐서

# 감정 분류 모델에 어휘 크기를 맞춰서 새로 생성
sent_model = SentimentAttentionClassifier(
    vocab_size=max(word_to_idx.values()) + 1,  # 어휘 크기
    embed_dim=32,                                # 임베딩 차원
    hidden_dim=64,                               # LSTM 은닉 차원
    num_classes=2                                 # 긍정/부정
)

# 추론 모드 (기울기 계산 불필요)
sent_model.eval()  # 평가 모드로 전환

# 순전파 실행
with torch.no_grad():  # 기울기 계산 비활성화 (추론 시 메모리 절약)
    pred_logits, pred_attn = sent_model(input_tensor)

# 예측 결과
pred_classes = pred_logits.argmax(dim=-1)  # 예측 클래스 (0 또는 1)
class_names = ["부정", "긍정"]               # 클래스 이름 매핑

# 각 문장별 시각화
for i, (sent, attn, cls) in enumerate(zip(sentences, pred_attn, pred_classes)):
    print(f"\n문장 {i+1}: {' '.join(sent)}")  # 문장 출력
    visualize_sentiment_attention(
        words=sent,                             # 단어 리스트
        attention_weights=attn,                 # Attention 가중치
        prediction=class_names[cls.item()],     # 예측 결과
        title=f"문장 {i+1}"                     # 제목
    )

print("\n참고: 학습되지 않은 모델이므로 Attention 가중치는 랜덤입니다.")
print("실제 학습 후에는 감정 단어(재미있고, 지루하고 등)에 높은 가중치가 부여됩니다.")

## 3.6 Seq2Seq + Attention 전체 순전파(Forward Pass) 및 Attention 시각화

Encoder와 Decoder를 연결하여 전체 Seq2Seq + Attention의 순전파를 실행하고,
각 Decoder 시점에서의 Attention 가중치를 히트맵으로 시각화합니다.

> **순전파(forward pass)**란? 입력 데이터가 모델을 통과하며 예측 결과를 만들어내는 과정입니다.
> 학습(training)에서는 이 뒤에 역전파(backward pass)가 이어지지만, 여기서는 추론(inference)만 수행합니다.

In [ ]:
# ============================================================
# Seq2Seq + Attention 전체 순전파 시뮬레이션
# Encoder → Decoder (매 시점 Attention) → 결과 시각화
# ============================================================

def seq2seq_inference(encoder, decoder, src_tokens, tgt_len, sos_idx=1):
    """Seq2Seq + Attention 모델의 추론(inference)을 수행합니다.
    
    Args:
        encoder: Encoder 모듈
        decoder: Decoder (Attention 내장) 모듈
        src_tokens: 소스 토큰 텐서 (1, src_len) - 배치 1
        tgt_len: 생성할 타겟 시퀀스 길이
        sos_idx: <SOS> 토큰의 인덱스 (기본값 1)
    
    Returns:
        predictions: 예측된 토큰 인덱스 리스트
        all_attention_weights: 각 시점의 Attention 가중치 (tgt_len, src_len)
    """
    # 평가 모드로 전환 (드롭아웃 비활성화)
    encoder.eval()  # Encoder 평가 모드
    decoder.eval()  # Decoder 평가 모드
    
    with torch.no_grad():  # 기울기 계산 비활성화
        # === Encoder 순전파 ===
        enc_outputs, hidden, cell = encoder(src_tokens)  # Encoder 실행
        
        # 첫 입력은 <SOS> 토큰
        input_token = torch.tensor([sos_idx])  # (1,) - <SOS> 인덱스
        
        # 결과를 저장할 리스트
        predictions = []       # 예측된 토큰 인덱스
        all_attn_weights = []  # 각 시점의 Attention 가중치
        
        # === Decoder 순전파 (매 시점) ===
        for t in range(tgt_len):  # 타겟 길이만큼 반복
            # Decoder 한 시점 실행
            pred, hidden, cell, attn_w = decoder.forward_step(
                input_token,    # 현재 시점 입력 토큰
                hidden,         # 이전 은닉 상태
                cell,           # 이전 셀 상태
                enc_outputs     # Encoder 전체 출력
            )
            
            # 가장 높은 확률의 토큰을 선택 (greedy decoding)
            top_token = pred.argmax(dim=-1)  # (1,) - 최고 점수 토큰
            
            # 결과 저장
            predictions.append(top_token.item())           # 토큰 인덱스 저장
            all_attn_weights.append(attn_w.squeeze(0).numpy())  # 가중치 저장
            
            # 다음 시점의 입력은 현재 예측한 토큰 (auto-regressive)
            input_token = top_token
    
    # Attention 가중치를 numpy 행렬로 변환
    all_attn_weights = np.array(all_attn_weights)  # (tgt_len, src_len)
    
    return predictions, all_attn_weights


# === Seq2Seq + Attention 전체 실행 ===
print("=" * 60)  # 구분선
print("Seq2Seq + Attention 전체 순전파 및 시각화")  # 제목
print("=" * 60)  # 구분선

# 가상의 소스 토큰 (배치 1, 길이 6)
fake_src_single = torch.randint(0, VOCAB_SIZE, (1, 6))  # 소스 시퀀스

# 전체 추론 실행
preds, attn_matrix = seq2seq_inference(
    encoder=encoder,         # 위에서 만든 Encoder
    decoder=decoder,         # 위에서 만든 Decoder
    src_tokens=fake_src_single,  # 소스 토큰
    tgt_len=4,               # 타겟 길이 4
    sos_idx=1                # <SOS> 토큰 인덱스
)

# 결과 출력
print(f"\n소스 토큰: {fake_src_single[0].tolist()}")      # 소스 토큰 리스트
print(f"예측 토큰: {preds}")                               # 예측된 토큰 리스트
print(f"Attention 행렬 크기: {attn_matrix.shape}")          # (4, 6)

# Attention 히트맵 시각화
src_labels = [f"src_{i}" for i in range(6)]    # 소스 라벨
tgt_labels = [f"tgt_{i}" for i in range(4)]    # 타겟 라벨

# 히트맵 그리기
plot_attention_heatmap(
    weights=attn_matrix,       # Attention 가중치 행렬
    src_labels=src_labels,     # 소스 단어 라벨
    tgt_labels=tgt_labels,     # 타겟 단어 라벨
    title="Seq2Seq + Bahdanau Attention 가중치 (PyTorch)"
)

print("참고: 학습되지 않은 모델이므로 Attention 패턴은 랜덤입니다.")
print("실제 학습 후에는 대각선에 가까운 패턴이 나타납니다 (단어 정렬).")

---

# Part 4: 정리

---

## 4.1 Attention의 한계

Attention은 RNN의 정보 병목 문제를 해결했지만, 여전히 근본적인 한계가 있습니다.

### 한계 1: $h_i$가 여전히 RNN이 만든 "뭉개진" 표현

```
입력:  "나는 어제 친구와 함께 서울역 근처 카페에서 커피를 마시며 이야기를 나누었다"
                                                              ↓
h_1    h_2    h_3    h_4    h_5    h_6    h_7    h_8    h_9    h_10   h_11   h_12
 ↑      ↑      ↑      ↑      ↑      ↑      ↑      ↑      ↑      ↑      ↑      ↑
RNN이 순차적으로 생성 → 앞쪽 h_i에는 "나는"의 정보만 있지 않음!
                       → h_5 = "함께"의 정보 + 이전 모든 정보의 뭉침
```

쉽게 말하면, **릴레이 달리기에서 바통을 넘기면서 정보가 섞이는 것**과 같습니다.
첫 주자의 메시지가 마지막 주자에게 도달하면 원래 내용이 변질되어 있습니다.

- $h_i$는 순수하게 $i$번째 단어의 표현이 아님
- RNN의 순차적 처리로 인해, $h_i$에는 **이전 시점의 정보가 섞여** 있음
- 양방향 LSTM을 쓰면 나아지지만, 근본적으로 "순서대로 읽는" 한계

### 한계 2: 병렬 처리(Parallelization) 불가

```
RNN:  h_1 → h_2 → h_3 → h_4 → h_5 (순차적! 앞이 끝나야 뒤를 계산)
      ↓
GPU의 수천 개 코어를 활용 불가 → 학습이 느림
```

쉽게 말하면, **생산 라인에서 한 명씩 순서대로 일하는 것** vs **여러 명이 동시에 일하는 것**의 차이입니다.

### 한계 3: 장거리 의존성 문제 완화, 하지만 해결 아님

- Attention은 Decoder가 먼 Encoder 시점을 **직접 참조**할 수 있게 해줌
- 하지만 Encoder의 $h_i$ 자체가 여전히 RNN으로 만들어지므로,
  $h_i$의 품질이 RNN의 장거리 의존성(long-range dependency) 문제에 영향 받음

## 4.2 Self-Attention / Transformer로의 발전 동기

위의 한계들을 해결하기 위한 아이디어:

> **"RNN을 아예 없애고, Attention만으로 시퀀스를 처리하면 어떨까?"**

이것이 바로 **Transformer** (Vaswani et al., 2017)의 핵심 아이디어입니다!

| 문제 | RNN + Attention | Transformer (Self-Attention Only) |
|------|----------------|----------------------------------|
| $h_i$의 품질 | RNN이 순차적으로 생성 (뭉개짐) | Self-Attention으로 **모든 단어를 직접 참조**하여 생성 |
| 병렬 처리 | RNN이 순차적 → 불가 | **완전 병렬 처리 가능** (모든 위치를 동시에 계산) |
| 장거리 의존성 | RNN 거쳐야 하므로 여전히 약함 | **경로 길이 = 1** (어떤 두 단어도 한 번에 연결) |
| 학습 속도 | 느림 | **매우 빠름** (GPU 활용 극대화) |

```
Transformer의 핵심:
  "Attention Is All You Need" - RNN, CNN 없이 Attention만으로!
  
  Self-Attention: 같은 시퀀스 내에서 모든 위치 쌍의 관계를 한 번에 계산
  Multi-Head Attention: 여러 관점에서 동시에 Attention (여러 개의 "형광펜")
  Positional Encoding: 순서 정보를 별도로 주입 (RNN의 순차 처리를 대체)
```

> 다음 챕터에서 Transformer를 밑바닥부터 구현합니다!

## 4.3 핵심 수식 요약표

이 노트북에서 다룬 모든 핵심 수식을 한 눈에 정리합니다.

### Attention 공통 프레임워크

| 단계 | 수식 | 설명 |
|------|------|------|
| **1. Score (점수)** | $e_{t,i} = \text{score}(s, h_i)$ | 관련도 점수 계산 |
| **2. Alignment (정렬)** | $\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_k \exp(e_{t,k})}$ | softmax로 확률 변환 |
| **3. Context (컨텍스트)** | $c_t = \sum_i \alpha_{t,i} h_i$ | 가중합으로 컨텍스트 생성 |

### Score 함수 비교

| 이름 | 수식 | 파라미터 수 | 특징 |
|------|------|------------|------|
| **Bahdanau (Additive, 가산)** | $e_{t,i} = v^T \tanh(W_s s_{t-1} + W_h h_i)$ | $d_a(d_s + d_h) + d_a$ | 가장 표현력이 높음, $s_{t-1}$ 사용 |
| **Luong Dot (내적)** | $e_{t,i} = s_t^T h_i$ | 0 | 가장 빠름, $d_s = d_h$ 필요 |
| **Luong General (일반)** | $e_{t,i} = s_t^T W_a h_i$ | $d_s \times d_h$ | 차원이 달라도 가능 |
| **Luong Concat (연결)** | $e_{t,i} = v^T \tanh(W_a [s_t; h_i])$ | $d_a(d_s + d_h) + d_a$ | Bahdanau와 유사 |
| **Scaled Dot-Product (스케일 내적)** | $e_{t,i} = \frac{s_t^T h_i}{\sqrt{d_k}}$ | 0 | Transformer에서 사용 |

### Self-Attention (Transformer)

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

| 기호 | 역할 | 비유 |
|------|------|------|
| $Q$ (Query, 질의) | "무엇을 찾는가" | 검색 질문 |
| $K$ (Key, 키) | "무엇을 제공하는가" | 문서 제목 |
| $V$ (Value, 값) | "실제 제공할 정보" | 문서 내용 |
| $\sqrt{d_k}$ (스케일링) | 분산 정규화 | 점수 스케일 맞추기 |

### 차원 요약

| 기호 | 의미 | 일반적인 값 |
|------|------|------------|
| $d_{\text{emb}}$ | 임베딩(embedding) 차원 | 128 ~ 512 |
| $d_h$ | Encoder 은닉(hidden) 차원 | 256 ~ 1024 |
| $d_s$ | Decoder 은닉(hidden) 차원 | 256 ~ 1024 |
| $d_a$ | Attention 은닉 차원 | 64 ~ 256 |
| $d_k$ | Key 차원 (Transformer) | 64 |

---

### 이 노트북에서 배운 핵심 포인트

1. **Attention은 정보 병목 문제를 해결합니다**: 고정된 컨텍스트 벡터 대신, 매 시점 다른 컨텍스트를 생성
2. **Score → Alignment → Context 3단계**: 모든 Attention은 이 프레임워크를 따릅니다
3. **Additive vs Dot-Product**: 표현력과 속도의 트레이드오프(trade-off)
4. **Attention의 한계**: RNN의 순차 처리 → Transformer로 발전

> 다음 챕터: **Transformer** (Self-Attention만으로 모든 것을!)